# Create Energimyndigheten Awards from SweCRIS

Creates Energimyndigheten (Swedish Energy Agency / Statens energimyndighet) awards from SweCRIS, Sweden's national research-grants registry. ~6K projects, 2008-2026.

**Prerequisites:**
- Run `scripts/local/energimyndigheten_to_s3.py` to download and upload the data first.

**Data source:** https://swecris-api.vr.se (SweCRIS API; the agency's own projektdatabas has no machine endpoint — see script header).
**S3 location:** `s3a://openalex-ingest/awards/energimyndigheten/energimyndigheten_projects.parquet`

**Energimyndigheten funder:**
- funder_id: 4320322711
- display_name: "Energimyndigheten"
- ROR: https://ror.org/0359z7n90
- DOI: 10.13039/501100004527

**Mapping notes:**
- `funder_award_id` = SweCRIS projectId with the `_Energi` suffix stripped (e.g. `P2023-00317`) — matches the grant number researchers cite.
- `amount` = `fundingsSek`, **SEK** (hardcoded: single-country funder), zeros treated as not-published.
- Energimyndigheten does not report PIs to SweCRIS (peopleList empty) → `lead_investigator` NULL by design.


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.energimyndigheten_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/energimyndigheten/energimyndigheten_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.energimyndigheten_raw;

In [ ]:
%sql
-- Step 1.5: inspect raw data before transforming
DESCRIBE openalex.awards.energimyndigheten_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.energimyndigheten_raw LIMIT 5;

In [ ]:
%sql
-- Step 1.6 funder existence check (Path A: F4320* must return exactly 1 row)
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320322711;

## Step 2: Create Energimyndigheten Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.energimyndigheten_awards
USING delta
AS
WITH
em_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320322711
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(REGEXP_REPLACE(TRIM(g.project_id), '_[A-Za-z]+$', ''))))) % 9000000000 as id,
        COALESCE(NULLIF(TRIM(g.title_english), ''), NULLIF(TRIM(g.title), '')) as display_name,
        COALESCE(NULLIF(TRIM(g.abstract_english), ''), NULLIF(TRIM(g.abstract), '')) as description,
        f.funder_id,
        REGEXP_REPLACE(TRIM(g.project_id), '_[A-Za-z]+$', '') as funder_award_id,
        NULLIF(TRY_CAST(g.amount AS DOUBLE), 0) as amount,
        'SEK' as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,
        CASE
            WHEN LOWER(COALESCE(g.type_of_award, '')) LIKE '%fellow%' THEN 'fellowship'
            WHEN LOWER(COALESCE(g.type_of_award, '')) LIKE '%stipend%' THEN 'fellowship'
            WHEN LOWER(COALESCE(g.type_of_award, '')) LIKE '%infrastructure%' THEN 'infrastructure'
            WHEN LOWER(COALESCE(g.type_of_award, '')) LIKE '%project%' THEN 'research'
            ELSE 'grant'
        END as funding_type,
        NULLIF(TRIM(g.type_of_award), '') as funder_scheme,
        'energimyndigheten' as provenance,
        TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
        YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
        YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
        CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as lead_investigator,
        CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
        CONCAT('https://www.vr.se/swecris#/project/', TRIM(g.project_id)) as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(REGEXP_REPLACE(TRIM(g.project_id), '_[A-Za-z]+$', ''))))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.energimyndigheten_raw g
    CROSS JOIN em_funder f
    WHERE g.project_id IS NOT NULL AND TRIM(g.project_id) != ''
)
SELECT * FROM awards_transformed;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'energimyndigheten' AND priority = 435;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    435 as priority  -- Energimyndigheten priority
FROM openalex.awards.energimyndigheten_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) as total_energimyndigheten_awards FROM openalex.awards.energimyndigheten_awards;

In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_abstract,
    COUNT(amount) as has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    COUNT(start_date) as has_start_date,
    MIN(amount) as min_amount,
    ROUND(AVG(amount), 0) as avg_amount,
    MAX(amount) as max_amount,
    ROUND(SUM(amount)/1e9, 2) as total_amount_billions_sek
FROM openalex.awards.energimyndigheten_awards;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.energimyndigheten_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 25;